In [1]:
"""
9.  Implement Naïve Bayes Classifier for Text classification use dataset of your choice from 
nltk corpus, preprocess it and use it for classification. Deal all the cases such as out of 
vocabulary, smoothing etc. Compare the results with library function of naïve bayes 
using relevant metrics
"""

'\n9.  Implement Naïve Bayes Classifier for Text classification use dataset of your choice from \nnltk corpus, preprocess it and use it for classification. Deal all the cases such as out of \nvocabulary, smoothing etc. Compare the results with library function of naïve bayes \nusing relevant metrics\n'

In [2]:
import numpy as np
import pandas as pd
import re, sklearn

In [3]:
from nltk.corpus import movie_reviews
data = [ (movie_reviews.raw(fileid), movie_reviews.categories(fileid)[0]) for fileid in movie_reviews.fileids() ]
df = pd.DataFrame(data, columns = ['text', 'category'] )
df

,text,category
0,"plot : two teen couples go to a church party ,...",neg
1,the happy bastard's quick movie review \ndamn ...,neg
2,it is movies like these that make a jaded movi...,neg
3,""" quest for camelot "" is warner bros . ' firs...",neg
4,synopsis : a mentally unstable man undergoing ...,neg
...,...,...
1995,wow ! what a movie . \nit's everything a movie...,pos
1996,"richard gere can be a commanding actor , but h...",pos
1997,"glory--starring matthew broderick , denzel was...",pos
1998,steven spielberg's second epic film on world w...,pos


In [4]:
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize 
from nltk.stem import WordNetLemmatizer

def lower(text):
    return text.lower()

def remove_punct(text):
    text = re.sub(r'[^\w\s]', '', text)
    text = re.sub(r'https?:\S+', '', text)
    return text

def tokenize(text):
    return word_tokenize(text)

def stop_words(tokens):
    stop_words = stopwords.words('english')
    return [ token for token in tokens if token not in stop_words and token.isalpha() ]

def lemmatization(tokens):
    lemmatizer = WordNetLemmatizer()
    return [ lemmatizer.lemmatize(token) for token in tokens ]


def preprocessing(text):
    text = lower(text)
    text = remove_punct(text)
    tokens = tokenize(text)
    tokens = stop_words(tokens)
    tokens = lemmatization(tokens)
    return ' '.join(tokens)

In [5]:
# Train Test Split
from sklearn.model_selection import train_test_split

train_x, test_x, train_y, test_y = train_test_split(df['text'], df['category'], test_size=0.2)
train_x.shape, train_y.shape, test_x.shape, test_y.shape

((1600,), (1600,), (400,), (400,))

In [6]:
# Scratch 
from collections import Counter

# Separate documents by class
positive_docs = [
    tokens for tokens, label in zip(train_x, train_y)
    if label == "pos"
]

negative_docs = [
    tokens for tokens, label in zip(train_x, train_y)
    if label == "neg"
]

# Vocabulary from training data only
vocabulary = set()

for tokens in train_x:
    vocabulary.update(tokens)

vocabulary = sorted(vocabulary)

print("Vocabulary size:", len(vocabulary))
print(vocabulary[:20])

Vocabulary size: 72
['\x05', '\n', '\x12', '\x13', '\x14', '\x16', ' ', '!', '"', '#', '$', '%', '&', "'", '(', ')', '*', '+', ',', '-']


In [7]:
positive_word_counts = Counter()
negative_word_counts = Counter()

for tokens in positive_docs:
    positive_word_counts.update(tokens)

for tokens in negative_docs:
    negative_word_counts.update(tokens)

print(positive_word_counts.most_common(10))
print(negative_word_counts.most_common(10))

[(' ', 619410), ('e', 301149), ('t', 226367), ('a', 202037), ('i', 189616), ('o', 180244), ('s', 174441), ('n', 170476), ('r', 147980), ('h', 133807)]
[(' ', 570653), ('e', 273183), ('t', 206963), ('a', 180246), ('i', 168603), ('o', 167878), ('s', 156234), ('n', 153715), ('r', 131000), ('h', 121739)]


In [8]:
total_documents = len(train_x)

positive_count = len(positive_docs)
negative_count = len(negative_docs)

positive_prior = positive_count / total_documents
negative_prior = negative_count / total_documents

print("P(positive):", positive_prior)
print("P(negative):", negative_prior)

P(positive): 0.491875
P(negative): 0.508125


In [9]:
positive_total_words = sum(positive_word_counts.values())
negative_total_words = sum(negative_word_counts.values())

vocab_size = len(vocabulary)

print("Positive words:", positive_total_words)
print("Negative words:", negative_total_words)
print("Vocabulary:", vocab_size)

Positive words: 3239281
Negative words: 2956283
Vocabulary: 72


In [10]:
def word_probability(word, word_counts, total_words):
    return (word_counts[word] + 1) / (total_words + vocab_size)

print(
    word_probability(
        "good",
        positive_word_counts,
        positive_total_words
    )
)

3.0870362075389745e-07


In [11]:
def predict_document(tokens):

    positive_probability = np.log(positive_prior)
    negative_probability = np.log(negative_prior)

    for word in tokens:

        # OOV handling
        if word not in vocabulary:
            continue

        positive_probability += np.log(
            word_probability(
                word,
                positive_word_counts,
                positive_total_words
            )
        )

        negative_probability += np.log(
            word_probability(
                word,
                negative_word_counts,
                negative_total_words
            )
        )

    if positive_probability > negative_probability:
        return "pos"

    return "neg"

In [12]:
predictions = []

for tokens in test_x:
    predictions.append(
        predict_document(tokens)
    )

print(predictions[:20])

['neg', 'pos', 'neg', 'pos', 'pos', 'neg', 'pos', 'neg', 'neg', 'neg', 'pos', 'neg', 'pos', 'pos', 'pos', 'neg', 'pos', 'pos', 'neg', 'pos']


In [13]:
from sklearn.metrics import classification_report

print(classification_report(test_y, predictions))

              precision    recall  f1-score   support

         neg       0.62      0.53      0.57       187
         pos       0.63      0.71      0.67       213

    accuracy                           0.63       400
   macro avg       0.63      0.62      0.62       400
weighted avg       0.63      0.63      0.62       400



In [14]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.preprocessing import LabelEncoder

train_x = train_x.apply(preprocessing)
test_x = test_x.apply(preprocessing)

encoder = LabelEncoder()
vectorizer = CountVectorizer()

train_x = vectorizer.fit_transform(train_x).toarray()
train_y = encoder.fit_transform(train_y)

In [15]:
# Vocabulary
vocabulary = vectorizer.get_feature_names_out()

test_x = vectorizer.transform(test_x).toarray()
test_y = encoder.transform(test_y)

vocabulary

array(['aaa', 'aaaaaaaaah', 'aaaaaaaahhhh', ..., 'zwicks', 'zwigoffs',
       'zycie'], shape=(37197,), dtype=object)

In [16]:
# Library Comparison
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import classification_report

model = MultinomialNB()
model.fit(train_x, train_y)

y_pred = model.predict(test_x)
print(classification_report(test_y, y_pred))

              precision    recall  f1-score   support

           0       0.79      0.82      0.80       187
           1       0.84      0.80      0.82       213

    accuracy                           0.81       400
   macro avg       0.81      0.81      0.81       400
weighted avg       0.81      0.81      0.81       400

